# SwarmBench: write your own agent and run it

An agent is a class with one method, `bid(obs) -> {"compute": .., "energy": .., "security": ..}`.
Everything in the `Observation` is public in the live arena:

| field | meaning |
|---|---|
| `round`, `total_rounds` | current round (1-based) and episode length |
| `budget` | this round's wallet *W* |
| `prices` | **last** round's clearing prices (total bid / capacity) |
| `capacities` | this round's pool sizes *C_k* |
| `profile` | own profile: `features` (battery, mobility, ...), `weights`, `q_min`, `s_min` |
| `rivals` | the public view of every other node (battery, mobility, weights, floors, ...) |
| `history` | own settled results, delivered with the live agent's one-round lag |

Shortcuts: `obs.battery`, `obs.mobility`, `obs.weights`, `obs.floors`.

The arena allocates *x = C·b/Σb* in every pool, scores a CES utility with service floors, and drains the battery
by 0.30(1+m)·x_E + 0.004 per round; at or below 0.05 the node sits out and recharges 0.22.

In [1]:
from IPython.display import Markdown, display

import swarmbench as sb
from swarmbench.report import markdown, summarize

## 1. A first agent

Bid the preference weights, but stop buying energy when the battery is low so the node does not run flat.

In [2]:
class BatterySaver(sb.Agent):
    """Weight bids; energy share cut linearly below a battery threshold."""

    def __init__(self, threshold: float = 0.4):
        self.threshold = threshold

    def bid(self, obs: sb.Observation) -> dict:
        w = dict(obs.weights)
        if obs.battery < self.threshold:
            w["energy"] *= max(0.0, obs.battery / self.threshold - 0.25)
        total = sum(w.values()) or 1.0
        return {k: obs.budget * w[k] / total for k in sb.RESOURCES}

## 2. Register it

`register_agent` puts it in the catalog so the modes, the reports and the command line (in this process) can use it.
Declare whether it is **realistic** (it is: it reads nothing about the bots) and its family.

In [3]:
for key in ("battery-saver", "battery-saver-0.6"):
    sb.CATALOG.pop(key, None)                      # lets the cell re-run
sb.register_agent("battery-saver", BatterySaver, name="Battery saver", family="Traditional (rules)",
                  realistic=True, description="Weight bids with an energy cut below 40% battery.")
sb.register_agent("battery-saver-0.6", lambda: BatterySaver(0.6), name="Battery saver (60%)",
                  family="Traditional (rules)", realistic=True, description="Threshold 0.6.")
print(sb.get_spec("battery-saver"))

AgentSpec(key='battery-saver', name='Battery saver', family='Traditional (rules)', order=1041, realistic=True, description='Weight bids with an energy cut below 40% battery.', factory=<class '__main__.BatterySaver'>, bot=False, aliases=(), note='', falls_back_to=None)


## 3. One episode, step by step

`Market.play` seats any participants and returns per-participant rows.

In [4]:
market = sb.Market()
seats = [sb.seat(sb.get_spec(b)) for b in ("naive-max", "even-split", "proportional")]
seats.append(sb.seat(sb.get_spec("battery-saver"), team="agent-battery-saver", baseline=False))
episode = market.play(70001, seats)
for row in episode.rows:
    print(f"{row['team']:22} score {row['score']:7.3f}  rank {row['rank']}  idle {row['idle']:2}  "
          f"{row['ms_mean']:.3f} ms/decision")
print("log social welfare", round(episode.lsw_total, 2), " welfare", round(episode.welfare, 2))

bot-naive-max          score  10.717  rank 3  idle 18  0.004 ms/decision
bot-even-split         score   8.141  rank 4  idle 19  0.003 ms/decision
bot-proportional       score  14.672  rank 1  idle 19  0.023 ms/decision
agent-battery-saver    score  13.050  rank 2  idle  3  0.007 ms/decision
log social welfare -269.6  welfare 46.58


## 4. Solo mode: against the three bots, next to reference agents

In [5]:
SEEDS = range(70001, 70006)
entries = ["template", "battery-saver", "battery-saver-0.6", "kelly"]
rows = sb.run_mode("solo", entries, SEEDS, workers=2)
summary = summarize(rows, "solo")
spec = {"seeds": len(SEEDS), "seed_start": SEEDS[0], "workers": 2}
display(Markdown(markdown(summary, spec, "solo", None)))

# Solo: each entry against the three supplied bots

Seeds 70001–70005 (5), `graded` scenario, 60 rounds, unchanged arena engine in-process. Every participant has the same device (minted from the seed and a fixed device label). Agents are listed in idea order: traditional rules, optimization, model-based planning, machine learning. **Realistic** = never uses the bots' names or code, so it can face unnamed opponents.

| # | Agent | Family | Realistic | Mean score | ±95% | Score rank | Δ vs cycle-aware (paired) | W/L vs cycle-aware | Beat all bots | Mean rank (of 4) | Idle | Floors | Exceptions | ms mean | ms max | Log social welfare | Bots' score | Welfare (Σ scores) |
|---:|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | Template: weight bids (`template`) | Traditional (rules) | yes | 12.794 | 0.922 | 4 | — | — | 0/5 | 2.40 | 19.2 | 3.00 | 0 | 0.719 | 12.6 | -185.8 | 45.46 | 58.25 |
| 2 | Guarded Kelly best response (`kelly`) | Optimization (one round) | yes | 17.035 | 1.285 | 1 | — | — | 5/5 | 1.00 | 5.0 | 1.60 | 0 | 4.3 | 8.6 | -200.9 | 37.77 | 54.80 |
| 3 | Battery saver (`battery-saver`) | Traditional (rules) | yes | 15.140 | 1.155 | 3 | — | — | 3/5 | 1.40 | 3.4 | 1.20 | 0 | 0.005 | 0.042 | -211.4 | 37.57 | 52.71 |
| 4 | Battery saver (60%) (`battery-saver-0.6`) | Traditional (rules) | yes | 15.453 | 1.297 | 2 | — | — | 5/5 | 1.00 | 1.6 | 1.40 | 0 | 0.006 | 0.039 | -214.8 | 36.30 | 51.75 |

Log social welfare is the arena's Σ_i log u_i summed over rounds for the 4-node market; welfare is the sum of all four final scores. Δ is the mean per-seed relative gain with a normal 95% interval; W/L counts seeds with a strictly higher/lower score.

Times are wall-clock milliseconds per decision measured inside the run (2 parallel workers), so they include contention; they are comparable across agents, not portable.


`cycle-aware` (the selected challenge submission) is the reference for the paired Δ column; it is not in this
small run, so that column shows "—". Add it to `entries` to compare against it.

## 5. Head-to-head and all-together

In [6]:
rows = sb.run_mode("head-to-head", ["battery-saver", "identified-robust", "naive-max"], range(70001, 70004),
                  workers=2)
summary = summarize(rows, "head-to-head")
for key, row in summary["_matrix"].items():
    print(f"{key:18}", {k: f"{v:.2f}" for k, v in row.items()})

battery-saver      {'identified-robust': '0.00', 'naive-max': '1.00'}
identified-robust  {'battery-saver': '1.00', 'naive-max': '1.00'}
naive-max          {'battery-saver': '0.00', 'identified-robust': '0.00'}


In [7]:
rows = sb.run_mode("all-together", ["bots", "battery-saver", "kelly", "identified-robust"], range(70001, 70004),
                  workers=2)
summary = summarize(rows, "all-together")
for key in sorted((k for k in summary if not k.startswith("_")), key=lambda k: -summary[k]["mean_score"]):
    s = summary[key]
    print(f"{key:18} {s['mean_score']:7.3f}  mean place {s['mean_rank']:.2f}  {s['ms_mean']:.3f} ms")

kelly                8.976  mean place 1.33  3.359 ms
identified-robust    8.928  mean place 1.67  161.320 ms
battery-saver        7.793  mean place 3.00  0.011 ms
naive-max            5.982  mean place 5.00  0.005 ms
even-split           5.629  mean place 4.67  0.003 ms
proportional         5.149  mean place 5.33  0.026 ms


## 6. Wrapping a plain strategy function

Functions with the signature of the built-in strategies, `f(budget, prices, capacities, profile, history)`,
plug in through `FunctionAgent`. `profile["_market"]["nodes"]` carries the public rivals.

In [8]:
def compute_heavy(budget, prices, capacities, profile, history):
    return {"compute": 0.6 * budget, "energy": 0.2 * budget, "security": 0.2 * budget}

sb.CATALOG.pop("compute-heavy", None)
sb.register_agent("compute-heavy", lambda: sb.FunctionAgent(compute_heavy), family="Traditional (rules)",
                  description="60/20/20 split.")
rows = sb.run_mode("solo", ["compute-heavy"], range(70001, 70004), workers=1)
print([round(r["score"], 3) for r in rows])

[15.284, 15.34, 15.313]


## 7. Using it from the command line

The command line reads the catalog of a fresh process. Put the class in a module that registers it and pass the
module with `--import`:

```python
# my_agents.py
import swarmbench as sb
from my_package import BatterySaver

sb.register_agent("battery-saver", BatterySaver, family="Traditional (rules)", realistic=True)
```

```bash
swarmbench --import my_agents --mode solo --agents battery-saver cycle-aware --seeds 20
```

Rules of thumb for a fair agent: keep per-episode state on `self` (one instance plays one episode), use only the
`Observation`, seed any randomness in `reset(seed)`, and stay well within the live arena's round window.